# Reinforcement Learning Basics — Teach an Agent to Play "Grid-Pac"

**AI Club Workshop Activity**

In the slides, we talked about the core ideas of reinforcement learning: agents, environments, states, actions, rewards, policies, exploration vs. exploitation, and the Q-learning update rule.

In this notebook you'll put those ideas into practice by training an agent to play a small Pac-Man-style game: navigate a maze, eat all the pellets, and avoid the ghost — using **Q-learning**.

### Before you start: make this notebook your own
This is a **shared, read-only copy**. Before you write any code:
1. Click **File → Save a copy in Drive** (you'll need to be signed in to a Google account).
2. Work in *your own copy* from here on — that way your edits and experiments are yours, and you won't affect anyone else's.

### How this notebook is organized
1. **The Game World** — a simplified Pac-Man environment (fully built for you)
2. **Watch it play randomly** — see the game before any learning happens
3. **Hyperparameters** — the knobs you'll experiment with
4. **The Q-table & Policy** — you fill in a small piece here
5. **The Q-learning Update Rule** — you fill in a small piece here too (this is the heart of the lesson)
6. **Train the agent**
7. **Watch your trained agent play**
8. **Experiments & discussion questions**

You do **not** need to write the game itself — that's provided so we can focus on the RL concepts. But there are a couple of short, clearly-marked `TODO` sections where you'll implement the actual learning logic using what we covered in the slides. If you get stuck, ask a facilitator or check the hints in the comments.

Run each cell top to bottom with **Shift+Enter**.

In [ ]:
import numpy as np
import random
import time
import matplotlib.pyplot as plt
from collections import defaultdict
from IPython.display import clear_output

random.seed(42)
np.random.seed(42)

%matplotlib inline
print("Libraries loaded. You're ready to go.")

## Part 1 — The Game World

Below is our maze. `#` is a wall, `.` is an open floor tile that starts with a pellet on it, and every other open tile is just walkable floor.

```
#######
#.....#
#.###.#
#.....#
#.#.#.#
#.....#
#######
```

- **Pac-agent** starts at the top of the maze.
- **Ghost** starts at the bottom of the maze and wanders — occasionally chasing Pac-agent (you can control how aggressively with `CHASE_PROBABILITY` later).
- The agent's **state** is: (its own position, the ghost's position, which pellets are still uneaten).
- The agent's **actions** are: up, down, left, right.
- The **reward** signal combines a small penalty per step (to encourage efficiency), a bonus for eating a pellet, a big bonus for clearing the board, and a big penalty for getting caught.

This is exactly the Agent / Environment / State / Action / Reward loop from the slides — just made concrete.

In [ ]:
MAZE = [
    "#######",
    "#.....#",
    "#.###.#",
    "#.....#",
    "#.#.#.#",
    "#.....#",
    "#######",
]

PACMAN_START = (1, 3)
GHOST_START = (5, 3)
PELLET_LOCATIONS = [(1, 1), (1, 5), (3, 3), (5, 1), (5, 5)]

ACTIONS = ["UP", "DOWN", "LEFT", "RIGHT"]
ACTION_DELTAS = {
    0: (-1, 0),   # UP
    1: (1, 0),    # DOWN
    2: (0, -1),   # LEFT
    3: (0, 1),    # RIGHT
}


class GridPacEnv:
    """A tiny Pac-Man-style grid world for learning Q-learning."""

    def __init__(self, step_penalty=-1, pellet_reward=10, win_reward=50,
                 ghost_penalty=-100, chase_probability=0.6, max_steps=150):
        self.grid = MAZE
        self.n_rows = len(self.grid)
        self.n_cols = len(self.grid[0])
        self.step_penalty = step_penalty
        self.pellet_reward = pellet_reward
        self.win_reward = win_reward
        self.ghost_penalty = ghost_penalty
        self.chase_probability = chase_probability
        self.max_steps = max_steps
        self.reset()

    def is_wall(self, pos):
        r, c = pos
        if r < 0 or r >= self.n_rows or c < 0 or c >= self.n_cols:
            return True
        return self.grid[r][c] == "#"

    def valid_moves_from(self, pos):
        moves = []
        for a, (dr, dc) in ACTION_DELTAS.items():
            new_pos = (pos[0] + dr, pos[1] + dc)
            if not self.is_wall(new_pos):
                moves.append(a)
        return moves

    def reset(self):
        self.pacman = PACMAN_START
        self.ghost = GHOST_START
        self.pellets = set(PELLET_LOCATIONS)
        self.steps = 0
        self.done = False
        return self.get_state()

    def get_state(self):
        return (self.pacman, self.ghost, tuple(sorted(self.pellets)))

    def _move_ghost(self):
        moves = self.valid_moves_from(self.ghost)
        if not moves:
            return
        if random.random() < self.chase_probability:
            # Chase: pick the valid move that gets closest (Manhattan distance) to Pac-agent
            def dist_after(a):
                dr, dc = ACTION_DELTAS[a]
                new_pos = (self.ghost[0] + dr, self.ghost[1] + dc)
                return abs(new_pos[0] - self.pacman[0]) + abs(new_pos[1] - self.pacman[1])
            best = min(moves, key=dist_after)
            action = best
        else:
            action = random.choice(moves)
        dr, dc = ACTION_DELTAS[action]
        self.ghost = (self.ghost[0] + dr, self.ghost[1] + dc)

    def step(self, action):
        if self.done:
            raise RuntimeError("Episode has ended. Call reset() before stepping again.")

        self.steps += 1
        reward = self.step_penalty

        dr, dc = ACTION_DELTAS[action]
        new_pos = (self.pacman[0] + dr, self.pacman[1] + dc)
        if not self.is_wall(new_pos):
            self.pacman = new_pos

        if self.pacman in self.pellets:
            self.pellets.remove(self.pacman)
            reward += self.pellet_reward

        if self.pacman == self.ghost:
            reward += self.ghost_penalty
            self.done = True
            info = {"result": "caught"}
            return self.get_state(), reward, self.done, info

        self._move_ghost()

        if self.pacman == self.ghost:
            reward += self.ghost_penalty
            self.done = True
            info = {"result": "caught"}
            return self.get_state(), reward, self.done, info

        if not self.pellets:
            reward += self.win_reward
            self.done = True
            info = {"result": "win"}
            return self.get_state(), reward, self.done, info

        if self.steps >= self.max_steps:
            self.done = True
            info = {"result": "timeout"}
            return self.get_state(), reward, self.done, info

        return self.get_state(), reward, self.done, {"result": "ongoing"}

    def render(self, title=""):
        """Draw the current state of the maze with matplotlib."""
        grid_colors = np.zeros((self.n_rows, self.n_cols))
        for r in range(self.n_rows):
            for c in range(self.n_cols):
                if self.grid[r][c] == "#":
                    grid_colors[r, c] = 0  # wall
                else:
                    grid_colors[r, c] = 1  # floor

        fig, ax = plt.subplots(figsize=(4, 4))
        cmap = plt.cm.colors.ListedColormap(["#1a1a2e", "#eaeaea"])
        ax.imshow(grid_colors, cmap=cmap, vmin=0, vmax=1)

        for (r, c) in self.pellets:
            ax.scatter(c, r, s=60, c="gold", marker="o", edgecolors="black", zorder=3)

        gr, gc = self.ghost
        ax.scatter(gc, gr, s=500, c="red", marker="o", zorder=4)
        ax.text(gc, gr, "G", ha="center", va="center", color="white", fontsize=12, fontweight="bold", zorder=5)

        pr, pc = self.pacman
        ax.scatter(pc, pr, s=500, c="#ffd400", marker="o", zorder=4)
        ax.text(pc, pr, "P", ha="center", va="center", color="black", fontsize=12, fontweight="bold", zorder=5)

        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f"{title}\nPellets left: {len(self.pellets)}  |  Step: {self.steps}")
        plt.show()
        plt.close(fig)


print("Environment defined. P = Pac-agent, G = Ghost, gold dots = pellets.")

## Part 2 — Watch it play randomly (before any learning)

Let's see what happens with **no intelligence at all** — the agent just takes random actions. This is our baseline: pure exploration, zero exploitation. Run it a couple of times and watch how badly (or luckily) it does.

In [ ]:
demo_env = GridPacEnv()
state = demo_env.reset()
demo_env.render("Random agent — before training")

for step in range(25):
    action = random.choice(range(4))
    state, reward, done, info = demo_env.step(action)
    clear_output(wait=True)
    demo_env.render(f"Random agent — before training (last reward: {reward})")
    time.sleep(0.3)
    if done:
        print("Episode ended:", info["result"])
        break

## Part 3 — Hyperparameters (your experiment dials)

These are the knobs from the slides: the learning rate, the discount factor, and the exploration schedule — plus some game-difficulty knobs. Leave them as-is for your first training run, then come back later and change them to see what happens (see the Experiments section at the end).

| Parameter | What it controls |
|---|---|
| `ALPHA` | Learning rate — how much each new experience updates the Q-table |
| `GAMMA` | Discount factor — how much the agent cares about future rewards vs. immediate ones |
| `EPSILON_START` | Initial exploration rate (chance of a random action) |
| `EPSILON_MIN` | The exploration rate never decays below this |
| `EPSILON_DECAY` | How quickly exploration shrinks toward `EPSILON_MIN` each episode |
| `NUM_EPISODES` | How many full games the agent plays while training |
| `MAX_STEPS` | Cap on steps per episode (prevents infinite wandering) |
| `CHASE_PROBABILITY` | How often the ghost actively chases vs. wanders randomly (game difficulty) |
| `PELLET_REWARD` / `GHOST_PENALTY` / `WIN_REWARD` / `STEP_PENALTY` | Reward shaping — how we tell the agent what "good" and "bad" mean |


In [ ]:
# ----- Tweak these! -----
ALPHA = 0.1              # learning rate
GAMMA = 0.9               # discount factor
EPSILON_START = 1.0       # start fully random
EPSILON_MIN = 0.05        # always keep a little exploration
EPSILON_DECAY = 0.999     # multiply epsilon by this after every episode

NUM_EPISODES = 4000
MAX_STEPS = 150

CHASE_PROBABILITY = 0.6   # 0 = ghost wanders randomly, 1 = ghost always chases

STEP_PENALTY = -1
PELLET_REWARD = 10
WIN_REWARD = 50
GHOST_PENALTY = -100
# -------------------------

print("Hyperparameters set.")

## Part 4 — The Q-table & the Policy (TODO)

Recall from the slides: a **Q-table** stores one value, `Q(state, action)`, for every state/action pair — an estimate of "how good is it to take this action from this state?" The agent's **policy** is simply: look up the Q-values for the current state and act on them.

We use a Python `defaultdict` so that any state we haven't seen yet automatically starts with Q-values of `[0, 0, 0, 0]` (one per action).

We also use **epsilon-greedy** action selection (straight from the slides):
- With probability `epsilon`, **explore**: take a random action.
- Otherwise, **exploit**: take the action with the highest known Q-value for this state.

The exploration branch is done for you. **Your job:** fill in the exploitation branch.

Hint: `q_table[state]` is a NumPy array of 4 numbers, one Q-value per action (`UP, DOWN, LEFT, RIGHT`). What NumPy function returns the *index* of the largest value in an array?

In [ ]:
q_table = defaultdict(lambda: np.zeros(4))


def choose_action(state, epsilon):
    if random.random() < epsilon:
        # Explore: take a random action
        return random.randint(0, 3)
    else:
        # TODO: Exploit! Return the action (0-3) with the highest Q-value
        # for this state. q_table[state] is a length-4 numpy array.
        #
        # Your code here (should be ~1 line):

        raise NotImplementedError("Fill in the exploitation branch above")


print("choose_action() defined — fill in the TODO before training.")

## Part 5 — The Q-learning Update Rule (TODO)

This is the one equation from the slides — the core of Q-learning:

\[ Q(s, a) \leftarrow Q(s, a) + \alpha \Big[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \Big] \]

In plain English: *"Nudge your current estimate of Q(s, a) toward the reward you actually got, plus a discounted estimate of the best you can do from here on out."*

Break it into 3 steps:
1. **`best_next_q`** — the best Q-value achievable from the *next* state (`0` if the episode just ended, since there's no future left).
2. **`target`** — what the Q-value "should" be: `reward + GAMMA * best_next_q`.
3. **Update** `q_table[state][action]` a fraction (`ALPHA`) of the way toward `target`.

Fill in the `TODO` in the training loop below.

In [ ]:
def train(num_episodes=NUM_EPISODES, verbose_every=500):
    env = GridPacEnv(
        step_penalty=STEP_PENALTY,
        pellet_reward=PELLET_REWARD,
        win_reward=WIN_REWARD,
        ghost_penalty=GHOST_PENALTY,
        chase_probability=CHASE_PROBABILITY,
        max_steps=MAX_STEPS,
    )
    epsilon = EPSILON_START
    episode_rewards = []
    results = []

    for episode in range(num_episodes):
        state = env.reset()
        total_reward = 0
        done = False

        while not done:
            action = choose_action(state, epsilon)
            next_state, reward, done, info = env.step(action)

            # TODO: Implement the Q-learning update rule shown above.
            #
            # 1. best_next_q = 0 if done else the max Q-value over all actions in next_state
            # 2. target = reward + GAMMA * best_next_q
            # 3. Update q_table[state][action] by moving it ALPHA of the way toward target
            #    (i.e. q_table[state][action] += ALPHA * (target - q_table[state][action]))
            #
            # Your code here (should be ~3 lines):

            raise NotImplementedError("Fill in the Q-learning update above")

            state = next_state
            total_reward += reward

        epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY)
        episode_rewards.append(total_reward)
        results.append(info["result"])

        if (episode + 1) % verbose_every == 0:
            recent = episode_rewards[-verbose_every:]
            recent_results = results[-verbose_every:]
            win_rate = recent_results.count("win") / len(recent_results)
            print(f"Episode {episode + 1}/{num_episodes} | "
                  f"avg reward (last {verbose_every}): {np.mean(recent):.1f} | "
                  f"win rate: {win_rate:.0%} | epsilon: {epsilon:.3f}")

    return episode_rewards, results


print("train() defined. Fill in both TODOs above, then run the next cell to train.")

## Part 6 — Train the agent

Once both TODOs above are filled in, run this cell. With the default settings this should take well under a minute and you should see the win rate climb as training progresses.

In [ ]:
episode_rewards, results = train()

## Part 7 — Learning curve

Did the agent actually get better? Plot total reward per episode (and a smoothed rolling average, since single-episode rewards are noisy).

In [ ]:
def moving_average(x, window=100):
    if len(x) < window:
        return np.array(x)
    return np.convolve(x, np.ones(window) / window, mode="valid")


plt.figure(figsize=(9, 4))
plt.plot(episode_rewards, alpha=0.3, label="Episode reward")
plt.plot(moving_average(episode_rewards, 100), label="Rolling average (100 episodes)", linewidth=2)
plt.axhline(0, color="gray", linewidth=0.8)
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.title("Learning curve")
plt.legend()
plt.show()

win_rate_overall = results.count("win") / len(results)
print(f"Overall win rate across training: {win_rate_overall:.1%}")
print(f"Win rate over the LAST 200 episodes: {results[-200:].count('win') / 200:.1%}")

## Part 8 — Watch your trained agent play

Now let's turn off exploration (epsilon = 0) and watch the agent act purely on what it learned.

In [ ]:
def watch_trained_agent(delay=0.35):
    env = GridPacEnv(
        step_penalty=STEP_PENALTY,
        pellet_reward=PELLET_REWARD,
        win_reward=WIN_REWARD,
        ghost_penalty=GHOST_PENALTY,
        chase_probability=CHASE_PROBABILITY,
        max_steps=MAX_STEPS,
    )
    state = env.reset()
    env.render("Trained agent")
    time.sleep(delay)

    done = False
    while not done:
        action = choose_action(state, epsilon=0.0)  # pure exploitation
        state, reward, done, info = env.step(action)
        clear_output(wait=True)
        env.render(f"Trained agent (last reward: {reward})")
        time.sleep(delay)

    print("Result:", info["result"])


watch_trained_agent()

## Part 9 — Experiments (do these!)

Now that you have a working agent, go back and change things. Re-run the hyperparameter cell, then re-run `train()`, the learning curve, and `watch_trained_agent()` to see the effect. Try at least 2-3 of these:

1. **Exploration schedule** — Set `EPSILON_DECAY = 0.99` (decays much faster) or `0.9999` (decays much slower). How does the learning curve change? Does the agent ever catch up either way?
2. **Discount factor** — Try `GAMMA = 0.1` (very short-sighted) vs `GAMMA = 0.99` (very far-sighted). Does the agent still learn to route toward far-away pellets?
3. **Learning rate** — Try `ALPHA = 0.9` (huge updates) vs `ALPHA = 0.01` (tiny updates). What happens to the stability of the learning curve?
4. **Reward shaping** — Make `STEP_PENALTY = 0` (no penalty for wandering). Does the agent still learn to be efficient, or does it start stalling?
5. **Reward shaping** — Make `GHOST_PENALTY` much smaller (e.g. `-5`) so getting caught barely matters. What happens to how cautious the agent is?
6. **Game difficulty** — Set `CHASE_PROBABILITY = 1.0` (ghost always chases) or `0.0` (ghost always wanders randomly). Is the game still winnable? Does the agent need more episodes?
7. **Training budget** — Cut `NUM_EPISODES` down to `500`. Does the agent get enough experience to learn a good policy?

### Discussion questions (tie back to the slides)
- Which hyperparameter had the *biggest* effect on how quickly the agent learned?
- Where did you see the exploration vs. exploitation trade-off show up concretely?
- What happened when you changed the rewards (Q4/Q5) — is this "reward shaping" and why does it matter so much for what the agent actually learns to do?
- Real-world RL systems (robotics, game AI, recommendation engines) can't use a Q-*table* like this — why not? (Hint: think about how many possible states a self-driving car or a Go board has.) What would you need to use instead? (This is a preview of a future topic: **Deep Q-Networks**.)
